In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
import sys
!{sys.executable} -m pip install torch torch-geometric biopython plotly numpy pandas scikit-learn --quiet
print("✓ Done")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
✓ Done


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os, io, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from pathlib import Path

from Bio.PDB import PDBParser, is_aa
from Bio.PDB import calc_dihedral

from torch_geometric.nn import TransformerConv, global_mean_pool, global_max_pool
from torch_geometric.data import Data
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")

display(HTML("""
<div style="
    background: linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
    border-radius:12px; padding:24px 32px;
    font-family:'Segoe UI',sans-serif; color:white;
    box-shadow:0 4px 20px rgba(0,0,0,0.4);">
  <h2 style="margin:0 0 8px 0; font-size:24px;">🧬 Protein Structure GNN Visualizer</h2>
  <p style="margin:0; color:#a0aec0; font-size:14px;">
      Drop any .pdb file → Graph Neural Network → Interactive 3D Dashboard
  </p>
</div>"""))
print(f"\n✓ Device  : {device}")
print(f"✓ PyTorch : {torch.__version__}")


✓ Device  : cpu
✓ PyTorch : 2.11.0+cu130


In [3]:
# ============================================================
# CELL 3 — Lookup tables
# ============================================================
AA_3TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    'MSE':'M','HSD':'H','HSE':'H','SEP':'S','TPO':'T',
    'CSO':'C','HIP':'H','HIE':'H','HID':'H',
}

AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
AA_IDX  = {a:i for i,a in enumerate(AA_LIST)}

AA_COLORS = {
    'A':'#FF8C00','V':'#FFA500','I':'#FF7F50','L':'#FF6347',
    'M':'#FF4500','F':'#FF8C00','W':'#DC143C','P':'#FF6347',
    'S':'#32CD32','T':'#3CB371','C':'#2E8B57','Y':'#228B22',
    'N':'#90EE90','Q':'#00FA9A','K':'#1E90FF','R':'#00BFFF',
    'H':'#87CEEB','D':'#FF1493','E':'#FF69B4',
    'G':'#DDA0DD','X':'#808080',
}

SS_COLORS  = {'H':'#FF4444','E':'#4488FF','C':'#44CC44','-':'#44CC44'}
SS_LABELS  = {'H':'α-Helix','E':'β-Strand','C':'Coil','-':'Coil'}

HYDROPHOBICITY = {
    'A': 1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C': 2.5,
    'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,'I': 4.5,
    'L': 3.8,'K':-3.9,'M': 1.9,'F': 2.8,'P':-1.6,
    'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V': 4.2,'X':0.0,
}

print("✓ Lookup tables ready")

✓ Lookup tables ready


In [4]:
# ============================================================
# CELL 4 — PDB parser (FINAL CLEAN MERGED VERSION)
# ============================================================

def _calc_phi_psi(residues):
    def vec(res, name):
        try:    return res[name].get_vector()
        except: return None

    out = []
    n   = len(residues)

    for i in range(n):
        phi = psi = 0.0

        if i > 0:
            atoms = [vec(residues[i-1],'C'), vec(residues[i],'N'),
                     vec(residues[i],'CA'), vec(residues[i],'C')]
            if all(a is not None for a in atoms):
                try: phi = calc_dihedral(*atoms)
                except: pass

        if i < n-1:
            atoms = [vec(residues[i],'N'), vec(residues[i],'CA'),
                     vec(residues[i],'C'), vec(residues[i+1],'N')]
            if all(a is not None for a in atoms):
                try: psi = calc_dihedral(*atoms)
                except: pass

        out.append((float(phi), float(psi)))

    return out


def _ss_from_angles(phi_psi):
    result = []
    for phi, psi in phi_psi:
        pd, sd = np.degrees(phi), np.degrees(psi)

        if -160 < pd < -40 and -70 < sd < 30:
            result.append('H')
        elif -160 < pd < -60 and (90 < sd <= 180 or -180 <= sd < -120):
            result.append('E')
        else:
            result.append('C')

    return result


def _try_dssp(structure, pdb_text):
    try:
        import tempfile
        from Bio.PDB import DSSP

        with tempfile.NamedTemporaryFile(suffix='.pdb', mode='w', delete=False) as tf:
            tf.write(pdb_text)
            tmp = tf.name

        dssp = DSSP(structure[0], tmp, dssp='mkdssp')
        os.unlink(tmp)

        out = {}
        for key, val in dssp:
            raw = val[2]
            if raw in ('H','G','I'): out[key] = 'H'
            elif raw in ('E','B'):   out[key] = 'E'
            else:                   out[key] = 'C'

        return out

    except Exception:
        return {}


class PDBParser3D:

    def __init__(self, contact_threshold: float = 8.0):
        self.threshold = contact_threshold
        self._biopdb   = PDBParser(QUIET=True)

    # ─────────────────────────────────────────────

    def parse_file(self, path: str, chain_id: str = None, chain_types: dict = None):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"PDB file not found: '{path}'")

        with open(path, 'r', errors='replace') as fh:
            pdb_text = fh.read()

        return self._parse(pdb_text, Path(path).stem, chain_id, chain_types)

    def parse_string(self, pdb_text: str, name='protein', chain_id=None, chain_types=None):
        return self._parse(pdb_text, name, chain_id, chain_types)

    # ─────────────────────────────────────────────

    def _parse(self, pdb_text, name, chain_id, chain_types):

        handle    = io.StringIO(pdb_text)
        structure = self._biopdb.get_structure(name, handle)
        model     = structure[0]

        all_chains = [c.id for c in model.get_chains()]

        # ── chain selection ───────────────────────
        if chain_id is None or chain_id.strip().lower() in ('all', 'auto', ''):
            chains = list(model.get_chains())
            chosen_id = 'all'
        else:
            selected = None
            for c in model.get_chains():
                if c.id.strip() == chain_id.strip():
                    selected = c
                    break

            if selected is None:
                print(f"  ⚠ Chain '{chain_id}' not found (available: {all_chains}) → using first chain")
                selected = list(model.get_chains())[0]

            chains = [selected]
            chosen_id = selected.id

        # ── collect residues ──────────────────────
        residues     = []
        chain_ids    = []
        chain_starts = []

        for chain in chains:
            chain_res = [r for r in chain.get_residues() if is_aa(r, standard=True)]
            if not chain_res:
                continue

            chain_starts.append(len(residues))
            residues.extend(chain_res)
            chain_ids.extend([chain.id] * len(chain_res))

        if len(residues) < 3:
            raise ValueError(f"Only {len(residues)} residues found in '{name}'")

        print(f"  Chain used     : '{chosen_id}' (available: {all_chains})")
        print(f"  Residues found : {len(residues)}")
        print(f"  Chains included: {[c.id for c in chains]}")

        # ── coords + B-factor ─────────────────────
        coords, bfactors = [], []

        for r in residues:
            ca = r['CA'] if 'CA' in r else None
            coords.append(ca.get_vector().get_array() if ca else np.zeros(3))
            bfactors.append(ca.get_bfactor() if ca else 0.0)

        coords   = np.array(coords, dtype=np.float32)
        bfactors = np.array(bfactors, dtype=np.float32)

        bf_norm = (bfactors - bfactors.mean()) / (bfactors.std() + 1e-8)

        # ── phi/psi per chain ─────────────────────
        phi_psi = []
        for chain in chains:
            chain_res = [r for r in chain.get_residues() if is_aa(r, standard=True)]
            phi_psi.extend(_calc_phi_psi(chain_res))

        # ── secondary structure ───────────────────
        dssp_map = _try_dssp(structure, pdb_text)

        ss_list = []
        for i, r in enumerate(residues):
            key = (r.get_parent().id, r.get_id())

            if key in dssp_map:
                ss_list.append(dssp_map[key])
            else:
                ss_list.append(_ss_from_angles([phi_psi[i]])[0])

        # ── chain types ───────────────────────────
        if chain_types is None:
            chain_types = {}

        resolved_types = [chain_types.get(cid, 'Protein') for cid in chain_ids]

        # ── dataframe ─────────────────────────────
        rows = []

        for i, r in enumerate(residues):
            res3 = r.get_resname().strip()
            aa   = AA_3TO1.get(res3, 'X')

            phi_r, psi_r = phi_psi[i]

            rows.append({
                'idx'        : i,
                'resname'    : res3,
                'aa'         : aa,
                'resnum'     : r.get_id()[1],
                'chain'      : chain_ids[i],
                'chain_type' : resolved_types[i],
                'x'          : float(coords[i,0]),
                'y'          : float(coords[i,1]),
                'z'          : float(coords[i,2]),
                'bfactor'    : float(bfactors[i]),
                'bfactor_n'  : float(bf_norm[i]),
                'hydro'      : HYDROPHOBICITY.get(aa, 0.0),
                'phi'        : float(np.degrees(phi_r)),
                'psi'        : float(np.degrees(psi_r)),
                'ss'         : ss_list[i],
                'color_aa'   : AA_COLORS.get(aa, '#808080'),
                'color_ss'   : SS_COLORS.get(ss_list[i], SS_COLORS['-']),
            })

        df = pd.DataFrame(rows)

        # ── contacts ──────────────────────────────
        edges = self._contacts(coords)

        # ── backbone bonds (per chain) ────────────
        bb_bonds = []

        for start, end in zip(chain_starts, chain_starts[1:] + [len(residues)]):
            for i in range(start, end - 1):
                bb_bonds.append((i, i + 1))

        print(f"  Contacts (≤{self.threshold}Å): {len(edges)}")

        ss_counts = df['ss'].value_counts().to_dict()
        print(f"  SS — H:{ss_counts.get('H',0)} E:{ss_counts.get('E',0)} C:{ss_counts.get('C',0)}")

        return dict(
            name=name,
            residues=df,
            coords=coords,
            edges=edges,
            bb_bonds=bb_bonds,
            chain=chosen_id,
            n_residues=len(df),
            chain_starts=chain_starts,
            chain_types=chain_types
        )

    # ─────────────────────────────────────────────

    def _contacts(self, coords):
        d  = coords[:, None, :] - coords[None, :, :]
        dm = np.linalg.norm(d, axis=-1)

        ii, jj = np.where((dm < self.threshold) & (dm > 0))
        return list(zip(ii.tolist(), jj.tolist()))


print("✓ PDB parser ready")

✓ PDB parser ready


In [5]:
# ============================================================
# CELL 5 — GNN
# ============================================================

def build_pyg_data(df: pd.DataFrame,
                   coords: np.ndarray,
                   edges: list) -> Data:
    """Convert parsed residue table → PyG Data object (27-dim nodes)."""
    n  = len(df)
    oh = np.zeros((n, 20), dtype=np.float32)
    for i, aa in enumerate(df['aa']):
        if aa in AA_IDX:
            oh[i, AA_IDX[aa]] = 1.0

    phi_r = np.radians(df['phi'].values)
    psi_r = np.radians(df['psi'].values)
    sc = np.stack([
        df['hydro'].values   / 5.0,
        df['bfactor_n'].values,
        np.sin(phi_r), np.cos(phi_r),
        np.sin(psi_r), np.cos(psi_r),
        np.arange(n)         / max(n-1, 1),
    ], axis=1).astype(np.float32)

    x   = np.concatenate([oh, sc], axis=1)    # [N, 27]
    src = [e[0] for e in edges]
    dst = [e[1] for e in edges]

    if src:
        diff = coords[src] - coords[dst]
        dist = np.linalg.norm(diff, axis=1, keepdims=True) / 8.0
        seq  = (np.abs(np.array(src)-np.array(dst)) == 1
                ).astype(np.float32)[:,None]
        ea   = np.concatenate([dist, seq], axis=1)
    else:
        ea   = np.zeros((0, 2), dtype=np.float32)

    return Data(
        x          = torch.FloatTensor(x),
        edge_index = torch.tensor([src, dst], dtype=torch.long),
        edge_attr  = torch.FloatTensor(ea),
        num_nodes  = n,
    )


class ProteinGNN(nn.Module):
    """
    Graph Transformer that encodes a protein structure graph.
    Returns:
        node_emb  [N, hidden]  — per-residue embeddings
        graph_emb [1, embed]   — whole-protein embedding
    """
    def __init__(self, node_dim=27, edge_dim=2,
                 hidden=128, embed=64,
                 n_layers=4, heads=4, dropout=0.1):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Linear(node_dim, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
        )
        self.convs = nn.ModuleList([
            TransformerConv(hidden, hidden//heads, heads=heads,
                            edge_dim=edge_dim, dropout=dropout,
                            concat=True)
            for _ in range(n_layers)
        ])
        self.norms  = nn.ModuleList([nn.LayerNorm(hidden)
                                      for _ in range(n_layers)])
        self.readout = nn.Sequential(
            nn.Linear(hidden*2, embed),
            nn.LayerNorm(embed), nn.GELU(),
        )

    def forward(self, data: Data):
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        batch = (data.batch
                 if hasattr(data,'batch') and data.batch is not None
                 else torch.zeros(x.size(0), dtype=torch.long,
                                  device=x.device))
        x = self.proj(x)
        for conv, norm in zip(self.convs, self.norms):
            x = norm(x + conv(x, ei, ea))
        g = self.readout(
            torch.cat([global_mean_pool(x, batch),
                       global_max_pool(x, batch)], dim=-1))
        return x, g


print("✓ GNN ready")

✓ GNN ready


In [6]:
# ============================================================
# CELL 6 — Visualizer (full, multi‑chain support for complexes)
# ============================================================

class Protein3DVisualizer:
    _DARK = '#0d1117'
    _GRID = '#1e2631'
    _TEXT = '#c9d1d9'

    def __init__(self, parsed: dict, node_emb: np.ndarray = None):
        self.parsed   = parsed
        self.df       = parsed['residues']
        self.coords   = parsed['coords']
        self.edges    = parsed['edges']
        self.name     = parsed['name']
        self.node_emb = node_emb

        # ── NEW: store chain start indices for multi‑chain backbone drawing
        self.chain_starts = parsed.get('chain_starts', None)

        if node_emb is not None and node_emb.shape[1] > 1:
            pca = PCA(n_components=3)
            self._pca3 = pca.fit_transform(node_emb)
            self._ev   = pca.explained_variance_ratio_
        else:
            self._pca3 = None
            self._ev   = None

    # ══════════════════════════════════════════════════════
    # PUBLIC  — same as before, unchanged
    # ══════════════════════════════════════════════════════

    def show_dashboard(self):
        """Full 6‑panel interactive dashboard (inline in Jupyter)."""

        fig = make_subplots(
            rows=2, cols=3,
            specs=[
                [{'type':'scatter3d'},{'type':'scatter3d'},{'type':'scatter3d'}],
                [{'type':'scatter'},  {'type':'bar'},       {'type':'heatmap'}],
            ],
            subplot_titles=[
                '① 3D Backbone — Secondary Structure',
                '② Contact Network',
                '③ GNN Node Embeddings (PCA)',
                '④ Ramachandran Plot',
                '⑤ Hydrophobicity Profile',
                '⑥ Embedding Heatmap',
            ],
            horizontal_spacing=0.03,
            vertical_spacing=0.10,
        )

        # row 1 — 3D panels
        self._backbone_traces(fig, 1, 1, 'ss')
        self._sphere_traces(fig, 1, 1, 'ss')
        self._contact_traces(fig, 1, 2)
        self._gnn_traces(fig, 1, 3)

        # row 2 — 2D panels
        self._rama_traces(fig, 2, 1)
        self._hydro_traces(fig, 2, 2)
        self._heatmap_trace(fig, 2, 3)

        # 3D scene styling
        for (r, c) in [(1,1),(1,2),(1,3)]:
            self._style_scene(fig, r, c)

        # 2D axes styling
        for (r, c) in [(2,1),(2,2),(2,3)]:
            fig.update_xaxes(gridcolor=self._GRID,
                             color=self._TEXT, row=r, col=c)
            fig.update_yaxes(gridcolor=self._GRID,
                             color=self._TEXT, row=r, col=c)

        fig.update_layout(
            title=dict(
                text=f"<b>🧬 {self.name}</b> — Protein Structure + GNN Analysis",
                font=dict(size=17, color='white'), x=0.5,
            ),
            height=980,
            paper_bgcolor=self._DARK,
            plot_bgcolor =self._DARK,
            font=dict(color=self._TEXT, size=11),
            showlegend=True,
            legend=dict(
                bgcolor='rgba(22,27,34,0.9)',
                bordercolor='#30363d', borderwidth=1,
                font=dict(color='white', size=10),
                x=1.01, y=0.98,
            ),
            margin=dict(l=5, r=120, t=70, b=5),
        )
        fig.show()

    # ── focused single‑view methods ───────────────────────

    def show_structure(self, color_by='ss'):
        fig = go.Figure()
        self._backbone_traces(fig, None, None, color_by)
        self._sphere_traces(fig, None, None, color_by)
        fig.update_layout(
            **self._base_layout(
                f"🧬 {self.name} — {color_by}", height=650
            ),
            scene=self._scene_dict(),
        )
        fig.show()

    def show_contact_network(self):
        fig = go.Figure()
        self._contact_traces(fig, None, None)
        fig.update_layout(
            **self._base_layout(f"🧬 {self.name} — Contact Network",
                                height=650),
            scene=self._scene_dict(),
        )
        fig.show()

    def show_gnn_embeddings(self):
        fig = go.Figure()
        self._gnn_traces(fig, None, None)
        fig.update_layout(
            **self._base_layout(f"🧬 {self.name} — GNN Embeddings",
                                height=650),
            scene=self._scene_dict(),
        )
        fig.show()

    def show_ramachandran(self):
        fig = go.Figure()
        self._rama_traces(fig, None, None)
        fig.update_xaxes(title='φ (°)', range=[-180,180],
                         gridcolor=self._GRID, color=self._TEXT,
                         zeroline=True, zerolinecolor='#555')
        fig.update_yaxes(title='ψ (°)', range=[-180,180],
                         gridcolor=self._GRID, color=self._TEXT,
                         zeroline=True, zerolinecolor='#555')
        fig.update_layout(
            **self._base_layout(f"🧬 {self.name} — Ramachandran",
                                height=550),
        )
        fig.show()

    def show_hydrophobicity(self):
        fig = go.Figure()
        self._hydro_traces(fig, None, None)
        fig.update_xaxes(title='Residue number',
                         gridcolor=self._GRID, color=self._TEXT)
        fig.update_yaxes(title='Kyte‑Doolittle score',
                         gridcolor=self._GRID, color=self._TEXT)
        fig.update_layout(
            **self._base_layout(f"🧬 {self.name} — Hydrophobicity",
                                height=400),
        )
        fig.show()

    def show_embedding_heatmap(self):
        fig = go.Figure()
        self._heatmap_trace(fig, None, None)
        fig.update_xaxes(title='Residue number',
                         color=self._TEXT)
        fig.update_yaxes(color=self._TEXT)
        fig.update_layout(
            **self._base_layout(f"🧬 {self.name} — Embedding Heatmap",
                                height=500),
        )
        fig.show()

    # ══════════════════════════════════════════════════════
    # TRACE BUILDERS  (shared between dashboard + singles)
    # ══════════════════════════════════════════════════════

    # ── NEW helper: iterate over chains ────────────────
    def _chain_data_iter(self):
        """Yield (start_idx, end_idx) for each chain in the flat residue list.
        If chain_starts is missing, treat the whole protein as one chain."""
        if self.chain_starts is None:
            yield (0, len(self.df))
        else:
            n = len(self.df)
            for i, start in enumerate(self.chain_starts):
                end = self.chain_starts[i+1] if i+1 < len(self.chain_starts) else n
                yield (start, end)

    # ── MODIFIED backbone traces (per chain) ────────────
    def _backbone_traces(self, fig, row, col, color_by):
        kw  = dict(row=row, col=col) if row else {}
        df  = self.df

        for start, end in self._chain_data_iter():
            chain_df = df.iloc[start:end]

            if color_by == 'ss':
                for ss, label in [('H','α-Helix'),('E','β-Strand'),('C','Coil')]:
                    sub = chain_df[chain_df['ss']==ss]
                    if sub.empty: continue
                    tr = go.Scatter3d(
                        x=sub['x'], y=sub['y'], z=sub['z'],
                        mode='lines',
                        line=dict(color=SS_COLORS[ss], width=6),
                        name=label, legendgroup=label,
                        hoverinfo='skip',
                        showlegend=(start == 0),   # only first chain shows legend
                    )
                    (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))
            else:
                colors, cmin, cmax, cscale, ctitle = self._color_cfg(color_by)
                line_kw = dict(width=5)
                if not isinstance(colors[0], str):
                    line_kw.update(
                        color=list(colors[start:end]),
                        colorscale=cscale,
                        cmin=cmin, cmax=cmax,
                    )
                else:
                    line_kw['color'] = list(colors[start:end])

                tr = go.Scatter3d(
                    x=chain_df['x'], y=chain_df['y'], z=chain_df['z'],
                    mode='lines', line=line_kw,
                    name='Backbone', hoverinfo='skip',
                    showlegend=(start == 0),
                )
                (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))

    # ── sphere traces (unchanged) ──────────────────────
    def _sphere_traces(self, fig, row, col, color_by):
        kw = dict(row=row, col=col) if row else {}
        df = self.df
        colors, cmin, cmax, cscale, ctitle = self._color_cfg(color_by)

        hover = [
            f"<b>{r['aa']}{r['resnum']}</b> ({r['resname']})<br>"
            f"SS: <b>{r['ss']}</b><br>"
            f"Hydrophobicity: {r['hydro']:.2f}<br>"
            f"φ: {r['phi']:.1f}°  |  ψ: {r['psi']:.1f}°<br>"
            f"B-factor: {r['bfactor']:.2f}"
            for _, r in df.iterrows()
        ]

        mk = dict(size=5, opacity=0.90,
                  line=dict(width=0.5, color='white'))

        if isinstance(colors, (list, np.ndarray)) and \
           not isinstance(colors[0], str):
            mk.update(color=list(colors), colorscale=cscale,
                      cmin=cmin, cmax=cmax,
                      colorbar=dict(
                          title=dict(text=ctitle,
                                     font=dict(color=self._TEXT)),
                          tickfont=dict(color=self._TEXT),
                          len=0.38, thickness=11,
                      ))
        else:
            mk['color'] = list(colors)

        tr = go.Scatter3d(
            x=df['x'], y=df['y'], z=df['z'],
            mode='markers', marker=mk,
            text=hover,
            hovertemplate='%{text}<extra></extra>',
            name='Residues', showlegend=False,
        )
        (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))

    # ── contact network (unchanged) ────────────────────
    def _contact_traces(self, fig, row, col):
        kw = dict(row=row, col=col) if row else {}
        df = self.df

        ex, ey, ez = [], [], []
        for i, j in self.edges:
            if abs(i-j) <= 1: continue
            ex += [df['x'].iloc[i], df['x'].iloc[j], None]
            ey += [df['y'].iloc[i], df['y'].iloc[j], None]
            ez += [df['z'].iloc[i], df['z'].iloc[j], None]

        for tr in [
            go.Scatter3d(x=ex, y=ey, z=ez, mode='lines',
                         line=dict(color='rgba(80,180,255,0.18)',
                                   width=1),
                         name='Contacts', hoverinfo='skip'),
            go.Scatter3d(x=df['x'], y=df['y'], z=df['z'],
                         mode='lines+markers',
                         line=dict(color='white', width=3),
                         marker=dict(size=3, color='white',
                                     opacity=0.55),
                         name='Chain', hoverinfo='skip',
                         showlegend=False),
        ]:
            (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))

    # ── GNN embeddings (MODIFIED: backbone per chain) ──
    def _gnn_traces(self, fig, row, col):
        kw = dict(row=row, col=col) if row else {}
        df = self.df

        # faint backbone lines – one trace per chain
        for start, end in self._chain_data_iter():
            chain_df = df.iloc[start:end]
            tr_line = go.Scatter3d(
                x=chain_df['x'], y=chain_df['y'], z=chain_df['z'],
                mode='lines',
                line=dict(color='rgba(255,255,255,0.2)', width=4),
                hoverinfo='skip', showlegend=False,
            )
            (fig.add_trace(tr_line, **kw) if kw else fig.add_trace(tr_line))

        # markers with embedding colouring (unchanged)
        if self._pca3 is not None:
            colors = self._pca3[:, 0]
            cscale = 'Plasma'
            ctitle = f'GNN PC1 ({self._ev[0]:.0%})'
            hover  = [
                f"<b>{r['aa']}{r['resnum']}</b><br>"
                f"Emb norm: {np.linalg.norm(self.node_emb[i]):.3f}<br>"
                f"PC1={self._pca3[i,0]:.3f}  "
                f"PC2={self._pca3[i,1]:.3f}"
                for i, (_, r) in enumerate(df.iterrows())
            ]
        else:
            colors = df['hydro'].values
            cscale = 'RdBu'
            ctitle = 'Hydrophobicity'
            hover  = [f"<b>{r['aa']}{r['resnum']}</b>"
                      for _, r in df.iterrows()]

        tr_mark = go.Scatter3d(
            x=df['x'], y=df['y'], z=df['z'],
            mode='markers',
            marker=dict(
                size=7,
                color=list(colors),
                colorscale=cscale,
                colorbar=dict(
                    title=dict(text=ctitle,
                               font=dict(color=self._TEXT)),
                    tickfont=dict(color=self._TEXT),
                    len=0.38, thickness=11,
                ),
                opacity=0.92,
                line=dict(width=0.5, color='white'),
            ),
            text=hover,
            hovertemplate='%{text}<extra></extra>',
            name='GNN Embedding',
        )
        (fig.add_trace(tr_mark, **kw) if kw else fig.add_trace(tr_mark))

    # ── Ramachandran (unchanged) ───────────────────────
    def _rama_traces(self, fig, row, col):
        kw  = dict(row=row, col=col) if row else {}
        df  = self.df
        clr = [SS_COLORS.get(s,'#aaa') for s in df['ss']]

        # allowed region backgrounds
        for x0,x1,y0,y1 in [(-160,-40,-70,10),
                              (-160,-60, 90,180),
                              (-160,-60,-180,-120)]:
            sh = dict(type='rect', x0=x0,x1=x1,y0=y0,y1=y1,
                      line=dict(color='rgba(255,255,255,0.2)',width=1),
                      fillcolor='rgba(255,255,255,0.04)')
            if row:
                fig.add_shape(**sh, row=row, col=col)
            else:
                fig.add_shape(**sh)

        tr = go.Scatter(
            x=df['phi'], y=df['psi'],
            mode='markers',
            marker=dict(color=clr, size=7, opacity=0.85,
                        line=dict(width=0.5,color='white')),
            text=[f"{r['aa']}{r['resnum']} ({r['ss']})"
                  for _,r in df.iterrows()],
            hovertemplate=(
                '<b>%{text}</b><br>'
                'φ = %{x:.1f}°<br>'
                'ψ = %{y:.1f}°'
                '<extra></extra>'
            ),
            name='Ramachandran',
        )
        (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))

        if row:
            fig.update_xaxes(title_text='φ (°)', range=[-180,180],
                             zeroline=True, zerolinecolor='#555',
                             gridcolor=self._GRID, row=row, col=col)
            fig.update_yaxes(title_text='ψ (°)', range=[-180,180],
                             zeroline=True, zerolinecolor='#555',
                             gridcolor=self._GRID, row=row, col=col)

    # ── hydrophobicity (unchanged) ─────────────────────
    def _hydro_traces(self, fig, row, col):
        kw  = dict(row=row, col=col) if row else {}
        df  = self.df
        clr = ['#FF4444' if h > 0 else '#4488FF' for h in df['hydro']]

        # ── bar plot ─────────────────────────────────────────
        tr = go.Bar(
            x=df['resnum'],
            y=df['hydro'],
            marker_color=clr,
            opacity=0.85,
            name='Hydrophobicity',
            text=[f"{r['aa']}{r['resnum']}" for _, r in df.iterrows()],
            hovertemplate='<b>%{text}</b><br>Score: %{y:.2f}<extra></extra>',
        )
        (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))

        # ── horizontal zero line ──
        line_style = dict(
            dash='dash',
            color='rgba(200,200,200,0.35)',
            width=1.5
        )

        if row:
            idx = (row - 1) * 3 + col
            xref = f"x{'' if idx == 1 else idx} domain"
            yref = f"y{'' if idx == 1 else idx}"
            fig.add_shape(
                type="line",
                x0=0, x1=1,
                y0=0, y1=0,
                xref=xref, yref=yref,
                line=line_style,
            )
            fig.update_xaxes(
                title_text='Residue number',
                gridcolor=self._GRID, color=self._TEXT,
                row=row, col=col
            )
            fig.update_yaxes(
                title_text='Hydrophobicity',
                gridcolor=self._GRID, color=self._TEXT,
                row=row, col=col
            )
        else:
            fig.add_shape(
                type="line",
                x0=min(df['resnum']), x1=max(df['resnum']),
                y0=0, y1=0,
                line=line_style,
            )

    # ── heatmap (unchanged) ────────────────────────────
    def _heatmap_trace(self, fig, row, col):
        kw = dict(row=row, col=col) if row else {}
        if self.node_emb is not None:
            ndim = min(24, self.node_emb.shape[1])
            mat  = self.node_emb[:, :ndim].T
            ylbl = [f'Dim {i}' for i in range(ndim)]
            ztxt = 'Embedding'
        else:
            mat  = np.stack([
                self.df['hydro'].values,
                self.df['bfactor_n'].values,
                np.sin(np.radians(self.df['phi'].values)),
                np.sin(np.radians(self.df['psi'].values)),
            ])
            ylbl = ['Hydrophobicity','B-factor','sin φ','sin ψ']
            ztxt = 'Feature'

        tr = go.Heatmap(
            z=mat, x=self.df['resnum'].tolist(), y=ylbl,
            colorscale='RdBu', zmid=0,
            colorbar=dict(
                title=dict(text=ztxt,
                           font=dict(color=self._TEXT)),
                tickfont=dict(color=self._TEXT),
                len=0.38, thickness=11,
            ),
            hovertemplate=(
                'Residue %{x}<br>%{y}: %{z:.3f}<extra></extra>'
            ),
            name='Heatmap',
        )
        (fig.add_trace(tr, **kw) if kw else fig.add_trace(tr))
        if row:
            fig.update_xaxes(title_text='Residue number',
                             color=self._TEXT, row=row, col=col)
            fig.update_yaxes(color=self._TEXT, row=row, col=col)

    # ── LAYOUT HELPERS (unchanged) ─────────────────────
    def _scene_dict(self):
        ax = dict(showgrid=False, zeroline=False,
                  showticklabels=False, title='',
                  backgroundcolor=self._DARK,
                  gridcolor=self._GRID)
        return dict(xaxis=ax, yaxis=ax, zaxis=ax,
                    bgcolor=self._DARK, aspectmode='data')

    def _style_scene(self, fig, row, col):
        idx = (row-1)*3 + col
        key = 'scene' if idx == 1 else f'scene{idx}'
        fig.update_layout(**{key: self._scene_dict()})

    def _base_layout(self, title, height=650):
        return dict(
            title=dict(text=f'<b>{title}</b>',
                       x=0.5, font=dict(color='white', size=15)),
            height=height,
            paper_bgcolor=self._DARK,
            plot_bgcolor =self._DARK,
            font=dict(color=self._TEXT),
            showlegend=True,
            legend=dict(bgcolor='rgba(22,27,34,0.9)',
                        bordercolor='#30363d', borderwidth=1,
                        font=dict(color='white')),
        )

    def _color_cfg(self, color_by):
        df = self.df
        if color_by == 'ss':
            return df['color_ss'].tolist(),None,None,None,'SS'
        if color_by == 'aa':
            return df['color_aa'].tolist(),None,None,None,'AA'
        if color_by == 'hydro':
            v = df['hydro'].values
            return v,-5.0,5.0,'RdBu','Hydrophobicity'
        if color_by == 'bfactor':
            v = df['bfactor'].values
            return v,float(v.min()),float(v.max()),'Viridis','B-factor'
        if color_by == 'embedding' and self._pca3 is not None:
            v = self._pca3[:,0]
            return v,float(v.min()),float(v.max()),'Plasma','GNN PC1'
        return df['color_aa'].tolist(),None,None,None,'AA'


print("✓ Visualizer ready (full, multi‑chain)") 

✓ Visualizer ready (full, multi‑chain)


In [7]:
# ============================================================
# CELL 7 — ← ONLY CELL YOU NEED TO EDIT
#           Put your PDB file path here and run
# ============================================================

# ┌─────────────────────────────────────────────────────┐
# │  Just change PDB_FILE to point to your .pdb file   │
# └─────────────────────────────────────────────────────┘

PDB_FILE = "/media/imil/New Volume/WORKSPACE/DATA_3D_MHC_TCR/TCR_complexes/9wbd.trunc.fit.pdb"

# Mapping: chain id → component type
CHAIN_TYPES = {
    'A': 'MHC',       # MHC heavy chain (class I)
    'C': 'Peptide',   # bound peptide
    'D': 'TCR',       # TCR α chain
    'E': 'TCR',       # TCR β chain
}

# (optional) leave CHAIN_ID = "all" to use all chains
CHAIN_ID = "all"

# ── auto-detect: if file missing, fall back to a demo download ───────
import urllib.request

if not os.path.isfile(PDB_FILE):
    print(f"'{PDB_FILE}' not found — downloading demo structure 1PPE …")
    url = "https://files.rcsb.org/download/1PPE.pdb"
    urllib.request.urlretrieve(url, "1PPE.pdb")
    PDB_FILE = "1PPE.pdb"
    CHAIN_ID = None

print(f"Using: {PDB_FILE}  |  chain: {CHAIN_ID or 'auto'}")

Using: /media/imil/New Volume/WORKSPACE/DATA_3D_MHC_TCR/TCR_complexes/9wbd.trunc.fit.pdb  |  chain: all


In [8]:
# ============================================================
# CELL 8 — Parse the PDB file (now with chain_types)
# ============================================================

pdb_parser = PDBParser3D(contact_threshold=8.0)

print(f"Parsing {PDB_FILE} …\n")
# note: we pass chain_types
parsed = pdb_parser.parse_file(PDB_FILE, chain_id=CHAIN_ID, chain_types=CHAIN_TYPES)

df     = parsed['residues']
coords = parsed['coords']
edges  = parsed['edges']

display(HTML(f"""
<div style="
    background:#161b22; border:1px solid #30363d;
    border-radius:8px; padding:16px 20px;
    font-family:monospace; color:#c9d1d9;">
  <b style="color:#58a6ff; font-size:15px;">🧬 {parsed['name']}</b>
  <br><br>
  Residues : <b>{parsed['n_residues']}</b> &nbsp;|&nbsp;
  Chain    : <b>{parsed['chain']}</b> &nbsp;|&nbsp;
  Contacts : <b>{len(edges)}</b><br>
  α-helix  : <b>{(df['ss']=='H').sum()}</b> &nbsp;
  β-strand : <b>{(df['ss']=='E').sum()}</b> &nbsp;
  Coil     : <b>{(df['ss']=='C').sum()}</b>
</div>"""))

Parsing /media/imil/New Volume/WORKSPACE/DATA_3D_MHC_TCR/TCR_complexes/9wbd.trunc.fit.pdb …

  Chain used     : 'all' (available: ['A', 'C', 'D', 'E'])
  Residues found : 404
  Chains included: ['A', 'C', 'D', 'E']
  Contacts (≤8.0Å): 4024
  SS — H:126 E:212 C:66


In [9]:
# ============================================================
# CELL 9 — Run GNN forward pass
# ============================================================

data  = build_pyg_data(df, coords, edges)
model = ProteinGNN(node_dim=27, edge_dim=2,
                   hidden=128, embed=64,
                   n_layers=4, heads=4).to(device)
model.eval()

with torch.no_grad():
    node_emb, graph_emb = model(data.to(device))

node_emb  = node_emb.cpu().numpy()
graph_emb = graph_emb.cpu().numpy()

n_params = sum(p.numel() for p in model.parameters())

display(HTML(f"""
<div style="
    background:#161b22; border:1px solid #30363d;
    border-radius:8px; padding:16px 20px;
    font-family:monospace; color:#c9d1d9;">
  <b style="color:#58a6ff;">⚡ GNN complete</b><br><br>
  Parameters  : <b>{n_params:,}</b><br>
  Node emb    : <b>{node_emb.shape}</b>
      &nbsp; [residues × hidden_dim]<br>
  Graph emb   : <b>{graph_emb.shape}</b>
      &nbsp; [1 × embed_dim]<br><br>
  Graph emb sample:
  <span style="color:#3fb950;">
    {np.array2string(graph_emb[0,:8], precision=4,
                     suppress_small=True)}
  </span>
</div>"""))

In [10]:
# ============================================================
# CELL 10 — Full interactive dashboard
# ============================================================
!pip install --upgrade -q nbformat
viz = Protein3DVisualizer(parsed, node_emb=node_emb)
viz.show_dashboard()


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [11]:
# ============================================================
# CELL 11 — Individual views
# ============================================================

# color_by options:
#   'ss'        → secondary structure  (red/blue/green)
#   'aa'        → amino acid identity
#   'hydro'     → hydrophobicity gradient (RdBu)
#   'bfactor'   → B-factor / flexibility (Viridis)
#   'embedding' → GNN PC1 projection (Plasma)

viz.show_structure(color_by='ss')

In [12]:
viz.show_contact_network()

In [13]:
viz.show_gnn_embeddings()

In [14]:
viz.show_embedding_heatmap()

In [15]:
viz.show_structure(color_by='hydro')

In [17]:
# ============================================================
# CELL 14 (or any number) — Coloured chain contact network
# ============================================================

import plotly.graph_objects as go
import numpy as np

# Colour maps
CHAIN_COLORS = {
    'MHC':     '#FF5555',   # red
    'Peptide': '#55FF55',   # green
    'TCR':     '#5555FF',   # blue
    'Protein': '#AAAAAA',   # fallback grey
}

INTERACTION_COLORS = {
    ('MHC', 'MHC'):       '#FFAAAA',
    ('MHC', 'Peptide'):   '#FFAA55',   # orange
    ('MHC', 'TCR'):       '#AA55FF',   # purple
    ('Peptide', 'Peptide'): '#AAFFAA',
    ('Peptide', 'TCR'):   '#55AAFF',   # cyan-ish
    ('TCR', 'TCR'):       '#AAAAFF',
    ('Protein', 'Protein'): '#CCCCCC',
}

# Extract existing data from the already-parsed structure
df      = parsed['residues']
coords  = parsed['coords']
edges   = parsed['edges']

# Build a colour array for nodes (one per residue)
node_colors = [CHAIN_COLORS.get(ct, CHAIN_COLORS['Protein'])
               for ct in df['chain_type']]

# Group edges by interaction type
edge_groups = {}
for i, j in edges:
    if abs(i - j) <= 1:
        continue                     # skip backbone bonds (shown separately)
    ct_a = df.iloc[i]['chain_type']
    ct_b = df.iloc[j]['chain_type']
    pair = tuple(sorted([ct_a, ct_b]))
    edge_groups.setdefault(pair, []).append((i, j))

# Create the figure
fig = go.Figure()

# ── Edges ────────────────────────────────────────────────
for pair, edgelist in edge_groups.items():
    ex, ey, ez = [], [], []
    for i, j in edgelist:
        ex += [df['x'].iloc[i], df['x'].iloc[j], None]
        ey += [df['y'].iloc[i], df['y'].iloc[j], None]
        ez += [df['z'].iloc[i], df['z'].iloc[j], None]

    color = INTERACTION_COLORS.get(pair, 'rgba(200,200,200,0.2)')
    name  = f"{pair[0]}–{pair[1]}"

    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez,
        mode='lines',
        line=dict(color=color, width=1.5),
        name=name,
        legendgroup=name,
        hoverinfo='skip',
    ))

# ── Nodes (spheres) coloured by chain type ───────────────
for ct, color in CHAIN_COLORS.items():
    idx = df[df['chain_type'] == ct].index
    if len(idx) == 0:
        continue
    fig.add_trace(go.Scatter3d(
        x=df.loc[idx, 'x'],
        y=df.loc[idx, 'y'],
        z=df.loc[idx, 'z'],
        mode='markers',
        marker=dict(size=4, color=color, opacity=0.9,
                    line=dict(width=1.5, color='white')),
        name=ct,
        legendgroup=ct,
        hoverinfo='skip',
    ))

# ── Backbone trace (white) ───────────────────────────────
fig.add_trace(go.Scatter3d(
    x=df['x'], y=df['y'], z=df['z'],
    mode='lines+markers',
    line=dict(color='white', width=2),
    marker=dict(size=2, color='white', opacity=0.2),
    name='Backbone',
    hoverinfo='skip',
))

# ── Layout styling ───────────────────────────────────────
fig.update_layout(
    title=dict(text=f"<b>🧬 {parsed['name']}</b> — Coloured Contact Network",
               x=0.5, font=dict(color='white', size=15)),
    height=650,
    paper_bgcolor='#0d1117',
    plot_bgcolor ='#0d1117',
    font=dict(color='#c9d1d9'),
    showlegend=True,
    legend=dict(bgcolor='rgba(22,27,34,0.9)',
                bordercolor='#30363d', borderwidth=1,
                font=dict(color='white')),
    scene=dict(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        aspectmode='data',
        bgcolor='#0d1117',
    ),
)

fig.show()

In [18]:
# ============================================================
# CELL – Coloured chain contact network + floating labels
# ============================================================

import plotly.graph_objects as go
import numpy as np

# Colour maps
CHAIN_COLORS = {
    'MHC':     '#FF5555',   # red
    'Peptide': '#55FF55',   # green
    'TCR':     '#5555FF',   # blue
    'Protein': '#AAAAAA',   # fallback grey
}

INTERACTION_COLORS = {
    ('MHC', 'MHC'):       '#FFAAAA',
    ('MHC', 'Peptide'):   '#FFAA55',   # orange
    ('MHC', 'TCR'):       '#AA55FF',   # purple
    ('Peptide', 'Peptide'): '#AAFFAA',
    ('Peptide', 'TCR'):   '#55AAFF',   # cyan-ish
    ('TCR', 'TCR'):       '#AAAAFF',
    ('Protein', 'Protein'): '#CCCCCC',
}

# Extract existing data from the already-parsed structure
df      = parsed['residues']
coords  = parsed['coords']
edges   = parsed['edges']

# Build a colour array for nodes
node_colors = [CHAIN_COLORS.get(ct, CHAIN_COLORS['Protein'])
               for ct in df['chain_type']]

# Group edges by interaction type
edge_groups = {}
for i, j in edges:
    if abs(i - j) <= 1:
        continue
    ct_a = df.iloc[i]['chain_type']
    ct_b = df.iloc[j]['chain_type']
    pair = tuple(sorted([ct_a, ct_b]))
    edge_groups.setdefault(pair, []).append((i, j))

# Create the figure
fig = go.Figure()

# ── Edges ────────────────────────────────────────────────
for pair, edgelist in edge_groups.items():
    ex, ey, ez = [], [], []
    for i, j in edgelist:
        ex += [df['x'].iloc[i], df['x'].iloc[j], None]
        ey += [df['y'].iloc[i], df['y'].iloc[j], None]
        ez += [df['z'].iloc[i], df['z'].iloc[j], None]

    color = INTERACTION_COLORS.get(pair, 'rgba(200,200,200,0.2)')
    name  = f"{pair[0]}–{pair[1]}"

    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez,
        mode='lines',
        line=dict(color=color, width=3),   # thicker edges
        name=name,
        legendgroup=name,
        hoverinfo='skip',
    ))

# ── Nodes (spheres) coloured by chain type ───────────────
for ct, color in CHAIN_COLORS.items():
    idx = df[df['chain_type'] == ct].index
    if len(idx) == 0:
        continue
    fig.add_trace(go.Scatter3d(
        x=df.loc[idx, 'x'],
        y=df.loc[idx, 'y'],
        z=df.loc[idx, 'z'],
        mode='markers',
        marker=dict(size=6, color=color, opacity=0.9,
                    line=dict(width=0.5, color='white')),
        name=ct,
        legendgroup=ct,
        hoverinfo='skip',
    ))

# ── Floating chain labels (MHC, TCR) ─────────────────────
# Compute a sensible vertical offset (10% of the z‑range)
z_range = df['z'].max() - df['z'].min()
z_offset = z_range * 0.1

for ct in ['MHC', 'TCR']:
    idx = df[df['chain_type'] == ct].index
    if len(idx) == 0:
        continue
    center = df.loc[idx, ['x','y','z']].mean()
    # Place label above the centroid
    fig.add_trace(go.Scatter3d(
        x=[center['x']],
        y=[center['y']],
        z=[center['z'] + z_offset],
        mode='text',
        text=[ct],
        textfont=dict(size=14, color=CHAIN_COLORS[ct], family='Arial, sans-serif'),
        showlegend=False,
        hoverinfo='skip',
    ))

# ── Backbone trace (white) ───────────────────────────────
fig.add_trace(go.Scatter3d(
    x=df['x'], y=df['y'], z=df['z'],
    mode='lines+markers',
    line=dict(color='white', width=2.5),
    marker=dict(size=2, color='white', opacity=0.4),
    name='Backbone',
    hoverinfo='skip',
))

# ── Layout styling ───────────────────────────────────────
fig.update_layout(
    title=dict(text=f"<b>🧬 {parsed['name']}</b> — Coloured Contact Network",
               x=0.5, font=dict(color='white', size=15)),
    height=700,
    paper_bgcolor='#0d1117',
    plot_bgcolor ='#0d1117',
    font=dict(color='#c9d1d9'),
    showlegend=True,
    legend=dict(bgcolor='rgba(22,27,34,0.9)',
                bordercolor='#30363d', borderwidth=1,
                font=dict(color='white')),
    scene=dict(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', backgroundcolor='#0d1117'),
        aspectmode='data',
        bgcolor='#0d1117',
    ),
)

fig.show()

In [19]:
# ============================================================
# CELL 14 — Conference-Grade Contact Network (FIXED)
# ============================================================

import plotly.graph_objects as go
import numpy as np

# ── Confirmed column names from diagnostic ──────────────────
CHAIN_COL  = 'chain'    # was 'chain_id'
RESNUM_COL = 'resnum'   # was 'res_num'

# ── Colour Palette ──────────────────────────────────────────
CHAIN_COLORS = {
    'MHC':     '#FF6B6B',
    'Peptide': '#51CF66',
    'TCR':     '#339AF0',
    'Protein': '#CED4DA',
}

INTERACTION_COLORS = {
    ('MHC',     'MHC'):     'rgba(255,107,107, 0.18)',
    ('MHC',     'Peptide'): 'rgba(255,180,50,  0.60)',
    ('MHC',     'TCR'):     'rgba(170,80,255,  0.50)',
    ('Peptide', 'Peptide'): 'rgba(81,207,102,  0.18)',
    ('Peptide', 'TCR'):     'rgba(50,200,255,  0.70)',
    ('TCR',     'TCR'):     'rgba(51,154,240,  0.18)',
    ('Protein', 'Protein'): 'rgba(200,200,200, 0.12)',
}

INTERACTION_WIDTHS = {
    ('MHC',     'Peptide'): 2.5,
    ('MHC',     'TCR'):     2.5,
    ('Peptide', 'TCR'):     3.5,   # thickest — most biologically critical
}

INTERACTION_LABELS = {
    ('MHC',     'MHC'):     'MHC internal',
    ('MHC',     'Peptide'): '⚡ MHC – Peptide',
    ('MHC',     'TCR'):     '⚡ MHC – TCR',
    ('Peptide', 'Peptide'): 'Peptide internal',
    ('Peptide', 'TCR'):     '⭐ Peptide – TCR',
    ('TCR',     'TCR'):     'TCR internal',
    ('Protein', 'Protein'): 'Protein internal',
}

# ── Pull parsed data ────────────────────────────────────────
df    = parsed['residues'].copy()
edges = parsed['edges']

print(f"✅ Residues      : {len(df)}")
print(f"✅ Edges         : {len(edges)}")
print(f"✅ Chain types   : {df['chain_type'].unique()}")
print(f"✅ Chains        : {sorted(df[CHAIN_COL].unique())}")

# ════════════════════════════════════════════════════════════
# 1. GROUP EDGES BY INTERACTION PAIR TYPE
# ════════════════════════════════════════════════════════════
edge_groups = {}
for i, j in edges:
    if abs(i - j) <= 1:
        continue                       # skip trivial backbone bonds
    ct_a = df.iloc[i]['chain_type']
    ct_b = df.iloc[j]['chain_type']
    pair = tuple(sorted([ct_a, ct_b]))
    edge_groups.setdefault(pair, []).append((i, j))

print(f"✅ Edge groups   : {list(edge_groups.keys())}")

# ════════════════════════════════════════════════════════════
# BUILD FIGURE
# ════════════════════════════════════════════════════════════
fig = go.Figure()

# ── 1. CONTACT EDGES ────────────────────────────────────────
for pair, edgelist in edge_groups.items():
    ex, ey, ez = [], [], []
    for i, j in edgelist:
        ex += [df['x'].iloc[i], df['x'].iloc[j], None]
        ey += [df['y'].iloc[i], df['y'].iloc[j], None]
        ez += [df['z'].iloc[i], df['z'].iloc[j], None]

    color = INTERACTION_COLORS.get(pair, 'rgba(200,200,200,0.15)')
    width = INTERACTION_WIDTHS.get(pair, 1.5)
    label = INTERACTION_LABELS.get(pair, f"{pair[0]}–{pair[1]}")

    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez,
        mode='lines',
        line=dict(color=color, width=width),
        name=label,
        legendgroup=label,
        legendrank=200,
        hoverinfo='skip',
    ))

# ── 2. COLOURED BACKBONE (per chain letter) ─────────────────
for ct, color in CHAIN_COLORS.items():
    sub = df[df['chain_type'] == ct]
    if sub.empty:
        continue

    for chain_val, grp in sub.groupby(CHAIN_COL):
        grp_s = grp.sort_values(RESNUM_COL)
        x_bb  = list(grp_s['x']) + [None]
        y_bb  = list(grp_s['y']) + [None]
        z_bb  = list(grp_s['z']) + [None]

        fig.add_trace(go.Scatter3d(
            x=x_bb, y=y_bb, z=z_bb,
            mode='lines',
            line=dict(color=color, width=3),
            name=f'{ct} backbone',
            legendgroup=ct,
            showlegend=False,
            hoverinfo='skip',
            opacity=0.45,
        ))

# ── 3a. GLOW HALOS (rendered before core so they sit behind) ─
for ct, color in CHAIN_COLORS.items():
    idx = df[df['chain_type'] == ct].index
    if len(idx) == 0:
        continue
    h = color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)

    fig.add_trace(go.Scatter3d(
        x=df.loc[idx, 'x'],
        y=df.loc[idx, 'y'],
        z=df.loc[idx, 'z'],
        mode='markers',
        marker=dict(
            size=5,
            color=f'rgba({r},{g},{b},0.07)',
            opacity=1.0,
        ),
        name=ct,
        legendgroup=ct,
        showlegend=False,
        hoverinfo='skip',
    ))

# ── 3b. RESIDUE CORE SPHERES ────────────────────────────────
for ct, color in CHAIN_COLORS.items():
    idx = df[df['chain_type'] == ct].index
    if len(idx) == 0:
        continue

    hover_parts = []
    for _, row in df.loc[idx].iterrows():
        tip = (
            f"<b>{row['chain_type']}</b>  ·  Chain {row[CHAIN_COL]}<br>"
            f"<b>{row['resname']}</b> ({row['aa']})  #{int(row[RESNUM_COL])}<br>"
            f"SS: {row['ss']}  |  φ {row['phi']:.1f}°  ψ {row['psi']:.1f}°<br>"
            f"Hydrophobicity: {row['hydro']}"
        )
        hover_parts.append(tip)

    fig.add_trace(go.Scatter3d(
        x=df.loc[idx, 'x'],
        y=df.loc[idx, 'y'],
        z=df.loc[idx, 'z'],
        mode='markers',
        marker=dict(
            size=5,
            color=color,
            opacity=0.95,
            line=dict(width=1, color='white'),
        ),
        name=ct,
        legendgroup=ct,
        legendrank=100,
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_parts,
    ))

# ── 4. CENTROID LABELS ──────────────────────────────────────
annotations = []
for ct, color in CHAIN_COLORS.items():
    sub = df[df['chain_type'] == ct]
    if sub.empty:
        continue
    cx = sub['x'].mean()
    cy = sub['y'].mean()
    cz = sub['z'].mean()
    annotations.append(dict(
        x=cx, y=cy, z=cz,
        text=f"<b>{ct}</b>",
        showarrow=False,
        font=dict(size=14, color=color, family='Arial Black'),
        bgcolor='rgba(13,17,23,0.70)',
        bordercolor=color,
        borderwidth=1.5,
        borderpad=5,
    ))

# ── 5. LAYOUT ───────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            f"<b>🧬 {parsed['name']}</b>  |  pMHC – TCR Contact Network"
            "<br><sup>Residue-level contacts · coloured by chain-pair interaction type"
            " · hover for residue details</sup>"
        ),
        x=0.5,
        font=dict(color='white', size=18, family='Arial'),
    ),
    height=780,
    paper_bgcolor='#0D1117',
    plot_bgcolor ='#0D1117',
    font=dict(color='#C9D1D9', family='Arial'),

    showlegend=True,
    legend=dict(
        x=0.01, y=0.98,
        bgcolor='rgba(22,27,34,0.93)',
        bordercolor='#30363D',
        borderwidth=1,
        font=dict(color='white', size=11),
        itemsizing='constant',
        tracegroupgap=5,
        title=dict(
            text='<b>Chain / Interaction</b>',
            font=dict(color='#8B949E', size=11),
        ),
    ),

    scene=dict(
        annotations=annotations,
        xaxis=dict(
            showgrid=False, zeroline=False,
            showticklabels=False, title='',
            backgroundcolor='#0D1117', showspikes=False,
        ),
        yaxis=dict(
            showgrid=False, zeroline=False,
            showticklabels=False, title='',
            backgroundcolor='#0D1117', showspikes=False,
        ),
        zaxis=dict(
            showgrid=False, zeroline=False,
            showticklabels=False, title='',
            backgroundcolor='#0D1117', showspikes=False,
        ),
        bgcolor='#0D1117',
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.6, y=0.8, z=0.6),
            up=dict(x=0,   y=0,   z=1),
        ),
    ),

    hoverlabel=dict(
        bgcolor='#161B22',
        bordercolor='#30363D',
        font=dict(color='white', size=12),
    ),

    margin=dict(l=0, r=0, t=95, b=0),
)

fig.show()

✅ Residues      : 404
✅ Edges         : 4024
✅ Chain types   : ['MHC' 'Peptide' 'TCR']
✅ Chains        : ['A', 'C', 'D', 'E']
✅ Edge groups   : [('MHC', 'MHC'), ('MHC', 'Peptide'), ('MHC', 'TCR'), ('Peptide', 'Peptide'), ('Peptide', 'TCR'), ('TCR', 'TCR')]
